---
**Pipeline:** [01 Dataset] → [02 Architecture] → [03 Optimization] → [04 Training] → **[05 Evaluation]** → [06 Export]

**Prev:** `04_model_training.ipynb` | **Next:** `06_model_export.ipynb`

---

# 📊 05 - Model Evaluation

## Benchmark on LFW, AgeDB, CFP-FP

This notebook evaluates the trained model on standard face verification benchmarks.

**Benchmarks:**
- 📷 LFW (Labeled Faces in the Wild)
- 👴 AgeDB (Age Database)
- 🔄 CFP-FP (Celebrities in Frontal-Profile)

**Target Metrics:**
- LFW: 98.5%+ accuracy
- AgeDB-30: 95%+ accuracy
- CFP-FP: 92%+ accuracy

---

## 1. Environment Setup

In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
    print("🌐 Running in Google Colab")
    PROJECT_ROOT = Path("/content/face_recognition")
    drive.mount('/content/drive')
    DRIVE_ROOT = Path("/content/drive/MyDrive/face_based_attendance_system")
    print(f"⚡ Data: {PROJECT_ROOT}")
    print(f"💾 Models: {DRIVE_ROOT}")
except ImportError:
    IN_COLAB = False
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DRIVE_ROOT = None
    print("💻 Running locally")

sys.path.insert(0, str(PROJECT_ROOT))

# Load checkpoints from Drive in Colab
if IN_COLAB and DRIVE_ROOT:
    CHECKPOINT_DIR = DRIVE_ROOT / "models" / "checkpoints"
else:
    CHECKPOINT_DIR = PROJECT_ROOT / "models" / "checkpoints"
DATA_DIR = PROJECT_ROOT / "data"

print(f"📂 Data: {DATA_DIR}")
print(f"💾 Checkpoints: {CHECKPOINT_DIR}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import roc_curve, auc, accuracy_score
from scipy import interpolate
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Device: {device}")

## 2. Load Model

In [ ]:
# Model definition (same as training)
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, p=1, g=1):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, k, s, p, groups=g, bias=False)
        self.bn = nn.BatchNorm2d(out_c)
        self.act = nn.PReLU(out_c)
    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

class DepthWise(nn.Module):
    def __init__(self, in_c, out_c, s=1):
        super().__init__()
        self.dw = nn.Conv2d(in_c, in_c, 3, s, 1, groups=in_c, bias=False)
        self.bn1 = nn.BatchNorm2d(in_c)
        self.act1 = nn.PReLU(in_c)
        self.pw = nn.Conv2d(in_c, out_c, 1, 1, 0, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.act2 = nn.PReLU(out_c)
    def forward(self, x):
        x = self.act1(self.bn1(self.dw(x)))
        return self.act2(self.bn2(self.pw(x)))

class DepthWiseRes(nn.Module):
    def __init__(self, in_c, out_c, s=1):
        super().__init__()
        self.residual = (s == 1 and in_c == out_c)
        self.dw = DepthWise(in_c, out_c, s)
    def forward(self, x):
        return x + self.dw(x) if self.residual else self.dw(x)

def make_stage(in_c, out_c, n, s=2):
    layers = [DepthWiseRes(in_c, out_c, s)]
    for _ in range(1, n):
        layers.append(DepthWiseRes(out_c, out_c, 1))
    return nn.Sequential(*layers)

class MobileFaceNet(nn.Module):
    def __init__(self, emb_dim=512):
        super().__init__()
        self.conv1 = ConvBlock(3, 64, 3, 2, 1)
        self.dw1 = DepthWise(64, 64, 1)
        self.stage1 = make_stage(64, 64, 5, 2)
        self.stage2 = make_stage(64, 128, 1, 2)
        self.stage3 = make_stage(128, 128, 6, 2)
        self.stage4 = make_stage(128, 128, 1, 1)
        self.conv_exp = ConvBlock(128, 512, 1, 1, 0)
        self.gdc = nn.Sequential(
            nn.Conv2d(512, 512, 7, 1, 0, groups=512, bias=False),
            nn.BatchNorm2d(512),
        )
        self.fc = nn.Linear(512, emb_dim, bias=False)
        self.bn = nn.BatchNorm1d(emb_dim)

    def forward(self, x):
        x = self.conv1(x)
        x = self.dw1(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.conv_exp(x)
        x = self.gdc(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        x = self.bn(x)
        return F.normalize(x, p=2, dim=1)

print("✅ Model class defined")

In [ ]:
# Load trained model
model = MobileFaceNet(emb_dim=512).to(device)

# Try to load best model
model_path = CHECKPOINT_DIR / 'best_backbone.pth'
if not model_path.exists():
    # Fall back to latest checkpoint
    ckpt_path = CHECKPOINT_DIR / 'checkpoint_latest.pth'
    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=device)
        # Extract backbone from full model
        backbone_state = {k.replace('backbone.', ''): v
                         for k, v in ckpt['model'].items()
                         if k.startswith('backbone.')}
        model.load_state_dict(backbone_state)
        print(f"✅ Loaded from checkpoint: {ckpt_path}")
    else:
        print("⚠️ No trained model found! Using random weights.")
else:
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"✅ Loaded best model: {model_path}")

model.eval()
print(f"📊 Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Preprocessing

In [ ]:
# Transform for evaluation
eval_transform = A.Compose([
    A.Resize(112, 112),
    A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ToTensorV2(),
])

def preprocess_image(img_path):
    """Load and preprocess image."""
    img = np.array(Image.open(img_path).convert('RGB'))
    img = eval_transform(image=img)['image']
    return img.unsqueeze(0)  # Add batch dimension

@torch.no_grad()
def get_embedding(model, img_path):
    """Extract embedding from image."""
    img = preprocess_image(img_path).to(device)
    emb = model(img)
    return emb.cpu().numpy().flatten()

@torch.no_grad()
def get_embeddings_batch(model, img_paths, batch_size=32):
    """Extract embeddings from batch of images."""
    embeddings = []

    for i in range(0, len(img_paths), batch_size):
        batch_paths = img_paths[i:i+batch_size]
        batch_imgs = torch.cat([preprocess_image(p) for p in batch_paths]).to(device)
        batch_embs = model(batch_imgs)
        embeddings.append(batch_embs.cpu().numpy())

    return np.vstack(embeddings)

print("✅ Preprocessing functions defined")

## 4. Evaluation Metrics

In [ ]:
# ============================================================================
# 3. TRAINING CONFIGURATION
# ============================================================================
print("📋 Setting up training configuration...")

from dataclasses import dataclass

@dataclass
class TrainingConfig:
    """Training configuration with GPU optimizations."""
    checkpoint_dir: Path = CHECKPOINT_DIR

    # Model
    embedding_dim: int = 512

    # ArcFace
    arcface_scale: float = 64.0
    arcface_margin: float = 0.5

    # ✅ Margin warmup: prevents 0% accuracy at start of training
    margin_warmup_epochs: int = 5
    arcface_margin_start: float = 0.0
    arcface_margin_end: float = 0.5

    # Training
    batch_size: int = 256 if torch.cuda.is_available() else 32  # ⬆️ OPTIMIZED: 256 for better GPU utilization
    num_epochs: int = 25
    # ✅ Lowered from 0.1 → 0.01 for stable ArcFace training from scratch
    base_lr: float = 0.01
    weight_decay: float = 5e-4
    momentum: float = 0.9

    # Schedule
    warmup_epochs: float = 1.0
    min_lr: float = 1e-6

    # Mixed Precision
    use_amp: bool = torch.cuda.is_available()
    grad_clip: float = 5.0

    # Checkpointing
    checkpoint_every_n_steps: int = 500
    keep_last_n_checkpoints: int = 3

    # Data Loading - OPTIMIZED FOR GPU UTILIZATION
    num_workers: int = 6 if IN_COLAB else 8  # ⬆️ OPTIMIZED: 6 for Colab (CPU limits), 8 for local
    pin_memory: bool = True
    prefetch_factor: int = 3  # ✨ NEW: 18 batches pre-loaded (6×3) or 24 (8×3)
    persistent_workers: bool = True  # ✨ NEW: Faster epoch transitions
    non_blocking: bool = True  # ✨ NEW: Async GPU transfers

config = TrainingConfig()

print("\n📋 Training Config:")
print(f"   Batch size: {config.batch_size}")
print(f"   Epochs: {config.num_epochs}")
print(f"   Base LR: {config.base_lr}")
print(f"   AMP: {config.use_amp}")
print(f"   ArcFace margin: {config.arcface_margin_start} → {config.arcface_margin_end} (warmup: {config.margin_warmup_epochs} epochs)")
print(f"   Checkpoint every: {config.checkpoint_every_n_steps} steps")
print(f"\n🚀 GPU Optimization Settings:")
print(f"   num_workers: {config.num_workers} (parallel data loading)")
print(f"   prefetch_factor: {config.prefetch_factor} ({config.num_workers * config.prefetch_factor} batches pre-loaded)")
print(f"   persistent_workers: {config.persistent_workers} (faster epochs)")
print(f"   non_blocking: {config.non_blocking} (async GPU transfers)")
print(f"   Expected GPU utilization: 85-95%+ ✅")

## 5. LFW Evaluation

In [ ]:
def load_lfw_pairs(pairs_file, lfw_dir):
    """
    Load LFW pairs from pairs.txt file.

    Format:
    - Same person: name n1 n2
    - Different: name1 n1 name2 n2
    """
    pairs = []
    labels = []

    if not pairs_file.exists():
        print(f"⚠️ Pairs file not found: {pairs_file}")
        return [], [], []

    with open(pairs_file) as f:
        lines = f.readlines()[1:]  # Skip header

    for line in lines:
        parts = line.strip().split()
        if len(parts) == 3:
            # Same person
            name, n1, n2 = parts
            img1 = lfw_dir / name / f"{name}_{int(n1):04d}.jpg"
            img2 = lfw_dir / name / f"{name}_{int(n2):04d}.jpg"
            label = 1
        elif len(parts) == 4:
            # Different people
            name1, n1, name2, n2 = parts
            img1 = lfw_dir / name1 / f"{name1}_{int(n1):04d}.jpg"
            img2 = lfw_dir / name2 / f"{name2}_{int(n2):04d}.jpg"
            label = 0
        else:
            continue

        if img1.exists() and img2.exists():
            pairs.append((img1, img2))
            labels.append(label)

    return pairs, labels

# Check for LFW data
LFW_DIR = DATA_DIR / "lfw" / "lfw-deepfunneled"
LFW_PAIRS = DATA_DIR / "lfw" / "pairs.txt"

if LFW_DIR.exists() and LFW_PAIRS.exists():
    pairs, labels = load_lfw_pairs(LFW_PAIRS, LFW_DIR)
    print(f"✅ LFW pairs loaded: {len(pairs)} pairs")
    print(f"   Positive: {sum(labels)}, Negative: {len(labels) - sum(labels)}")
else:
    print("⚠️ LFW dataset not found")
    print(f"   Expected: {LFW_DIR}")
    print("\n📥 To download LFW:")
    print("   Run: !wget http://vis-www.cs.umass.edu/lfw/lfw-deepfunneled.tgz")
    pairs, labels = [], []

In [ ]:
# Evaluate on LFW
if pairs:
    print("\n🔍 Evaluating on LFW...")

    # Extract embeddings
    emb1_list = []
    emb2_list = []

    for img1, img2 in tqdm(pairs, desc="Extracting embeddings"):
        e1 = get_embedding(model, img1)
        e2 = get_embedding(model, img2)
        emb1_list.append(e1)
        emb2_list.append(e2)

    embeddings1 = np.array(emb1_list)
    embeddings2 = np.array(emb2_list)
    labels_array = np.array(labels)

    # Evaluate
    results = evaluate_pairs(embeddings1, embeddings2, labels_array)

    print("\n📊 LFW Results:")
    print("="*40)
    print(f"   Accuracy: {results['accuracy']:.2f}%")
    print(f"   AUC: {results['auc']:.4f}")
    print(f"   Best Threshold: {results['threshold']:.2f}")
    print(f"   TAR@FAR=0.001: {results['tar@far0.001']:.2f}%")
    print("="*40)

    lfw_results = results
else:
    print("\n⚠️ Skipping LFW evaluation (no data)")
    lfw_results = None

## 6. Test on Sample Images

In [ ]:
# Create synthetic test if no LFW data
TEST_DIR = DATA_DIR / "test_images"

def test_on_samples(test_dir):
    """Test model on sample images."""
    if not test_dir.exists():
        print(f"⚠️ Test directory not found: {test_dir}")
        return

    # Find all images
    images = list(test_dir.glob('*.jpg')) + list(test_dir.glob('*.png'))

    if len(images) < 2:
        print("⚠️ Need at least 2 images for comparison")
        return

    print(f"\n🔍 Testing on {len(images)} images from {test_dir}")

    # Extract embeddings
    embeddings = {}
    for img_path in images:
        emb = get_embedding(model, img_path)
        embeddings[img_path.name] = emb
        print(f"   ✅ {img_path.name}: embedding shape {emb.shape}")

    # Compare all pairs
    print("\n📊 Similarity Matrix:")
    names = list(embeddings.keys())

    for i, n1 in enumerate(names):
        for j, n2 in enumerate(names):
            if i < j:
                sim = compute_cosine_similarity(embeddings[n1], embeddings[n2])
                match = "✓ MATCH" if sim > 0.5 else "✗ NO MATCH"
                print(f"   {n1} vs {n2}: {sim:.4f} {match}")

test_on_samples(TEST_DIR)

## 7. ROC Curve Visualization

In [ ]:
# Plot ROC curve
if lfw_results:
    plt.figure(figsize=(10, 8))

    plt.plot(lfw_results['fpr'], lfw_results['tpr'],
             label=f"LFW (AUC={lfw_results['auc']:.4f})")
    plt.plot([0, 1], [0, 1], 'k--', label='Random')

    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve - Face Verification')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)

    # Add annotations
    plt.annotate(f'Accuracy: {lfw_results["accuracy"]:.2f}%',
                 xy=(0.6, 0.2), fontsize=12)
    plt.annotate(f'Threshold: {lfw_results["threshold"]:.2f}',
                 xy=(0.6, 0.15), fontsize=12)

    plt.tight_layout()
    plt.savefig(CHECKPOINT_DIR / 'roc_curve.png', dpi=150)
    plt.show()
else:
    print("No results to plot")

## 8. Embedding Visualization

In [ ]:
# Visualize embedding distribution
if pairs and len(pairs) > 0:
    # Get sample embeddings
    sample_embs = embeddings1[:100] if len(embeddings1) >= 100 else embeddings1

    # Compute similarity distribution
    same_sims = []
    diff_sims = []

    for i, (e1, e2) in enumerate(zip(embeddings1, embeddings2)):
        sim = compute_cosine_similarity(e1, e2)
        if labels[i] == 1:
            same_sims.append(sim)
        else:
            diff_sims.append(sim)

    plt.figure(figsize=(10, 5))

    plt.hist(same_sims, bins=50, alpha=0.7, label='Same Person', color='green')
    plt.hist(diff_sims, bins=50, alpha=0.7, label='Different Person', color='red')

    if lfw_results:
        plt.axvline(lfw_results['threshold'], color='blue', linestyle='--',
                   label=f'Threshold ({lfw_results["threshold"]:.2f})')

    plt.xlabel('Cosine Similarity')
    plt.ylabel('Count')
    plt.title('Similarity Distribution')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(CHECKPOINT_DIR / 'similarity_distribution.png', dpi=150)
    plt.show()
else:
    print("No data for visualization")

## 9. Inference Speed Benchmark

In [ ]:
import time

def benchmark_inference(model, num_iterations=100, batch_size=1):
    """Benchmark inference speed."""
    model.eval()

    # Warmup
    dummy = torch.randn(batch_size, 3, 112, 112).to(device)
    for _ in range(10):
        with torch.no_grad():
            _ = model(dummy)

    if device.type == 'cuda':
        torch.cuda.synchronize()

    # Benchmark
    start = time.time()
    for _ in range(num_iterations):
        with torch.no_grad():
            _ = model(dummy)

    if device.type == 'cuda':
        torch.cuda.synchronize()

    total_time = time.time() - start
    avg_time = total_time / num_iterations * 1000  # ms

    return avg_time

# Run benchmark
print("\n⚡ Inference Speed Benchmark:")
print("="*40)

for batch_size in [1, 8, 32]:
    avg_time = benchmark_inference(model, num_iterations=100, batch_size=batch_size)
    throughput = batch_size / (avg_time / 1000)
    print(f"   Batch {batch_size}: {avg_time:.2f}ms ({throughput:.1f} img/s)")

print("="*40)

## 10. Summary

In [ ]:
print("""
📋 Evaluation Summary
="*50

Model: MobileFaceNet
Parameters: ~1M
Embedding: 512-D, L2-normalized
""")

if lfw_results:
    print(f"""
LFW Results:
  Accuracy: {lfw_results['accuracy']:.2f}%
  AUC: {lfw_results['auc']:.4f}
  Best Threshold: {lfw_results['threshold']:.2f}
  TAR@FAR=0.001: {lfw_results['tar@far0.001']:.2f}%
""")

print("""
="*50

📋 Next Steps:
  1. Run 07_model_export.ipynb for ONNX export
  2. Deploy to production!
  3. Fine-tune if needed
""")